In [1]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

sports = ["cricket", "soccer", "rugby"]
docs_root = Path("sports_docs")

all_docs = []
for sport in sports:
    sport_dir = docs_root / sport
    reader = SimpleDirectoryReader(input_dir=str(sport_dir))
    docs = reader.load_data()
    for d in docs:
        d.metadata["sport"] = sport
        d.metadata["content_type"] = Path(d.metadata.get("file_name", "")).stem
    all_docs.extend(docs)
    print(f"{sport}: loaded {len(docs)} document(s)")

print(f"\nTotal documents loaded: {len(all_docs)}")

cricket: loaded 93 document(s)
soccer: loaded 70 document(s)
rugby: loaded 33 document(s)

Total documents loaded: 196


In [2]:
from collections import Counter

counts = Counter((d.metadata["sport"], d.metadata["content_type"]) for d in all_docs)
for key, count in sorted(counts.items()):
    print(f"{key}: {count} page(s)")

('cricket', 'history'): 19 page(s)
('cricket', 'records'): 58 page(s)
('cricket', 'rules'): 16 page(s)
('rugby', 'history'): 15 page(s)
('rugby', 'rules'): 18 page(s)
('soccer', 'history'): 31 page(s)
('soccer', 'records'): 27 page(s)
('soccer', 'rules'): 12 page(s)


In [3]:
records_doc = next(d for d in all_docs if d.metadata["content_type"] == "records")
rules_doc = next(d for d in all_docs if d.metadata["content_type"] == "rules")

print("=== RULES sample ===")
print(rules_doc.metadata)
print(rules_doc.text[:500])

print("\n=== RECORDS sample ===")
print(records_doc.metadata)
print(records_doc.text[:500])

=== RULES sample ===
{'page_label': '1', 'file_name': 'rules.pdf', 'file_path': 'c:\\Users\\koolr\\OneDrive\\Agentic AI June 2026\\Week 2 Project - RAG\\sports_rag_starter\\sports_rag\\sports_docs\\cricket\\rules.pdf', 'file_type': 'application/pdf', 'file_size': 1083038, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14', 'sport': 'cricket', 'content_type': 'rules'}
Laws of Cricket
The Laws of Cricket is a code that specifies the rules of the game of cricket worldwide. The earliest
known code was drafted in 1744. Since 1788, the code has been owned and maintained by the private
Marylebone Cricket Club (MCC) in Lord's Cricket Ground, London. There are currently 42 Laws
(always written with a capital "L"), which describe all aspects of how the game is to be played. MCC
has re-coded the Laws six times, each with interim revisions that produce more than one edi

=== RECORDS sample ===
{'page_label': '1', 'file_name': 'records.pdf', 'file_path': 'c:\\Users\\koolr\\OneDrive\\

In [4]:
cricket_records = [d for d in all_docs if d.metadata["sport"]=="cricket" and d.metadata["content_type"]=="records"]
print(f"Found {len(cricket_records)} cricket records pages")

sample = cricket_records[15]
print("Page:", sample.metadata.get("page_label"))
print(sample.text[:800])

Found 58 cricket records pages
Page: 16
Glenn Maxwell of Australia (pictured) has
the highest strike rate at the World Cup
with 160.32.[e][76]
Highest strike rate
Rank Average Player Team Runs Balls faced Period
1 160.32 Glenn Maxwell
  Australia 901 562 2015–2023
2 120.84 Brendon McCullum
  New Zealand 742 614 2003–2015
3 118.20 Jos Buttler
  England 591 500 2015–2023
4 117.29 AB de Villiers
  South Africa 1,207 1,029 2007–2015
5 115.14 Kapil Dev
  India 669 581 1979–1992
Qualification: 500 balls faced
Last updated: 19 November 2023[76]
A half-century is a score of between 50 and 99 runs. Statistically, once a batsman's score reaches 100,
it is no longer considered a half-century but a century.[78]
Sachin Tendulkar of India has scored the most half-centuries at the Cricket World Cup with 15. He is
followed by India's Virat Kohl


In [5]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(all_docs)

print(f"Total chunks: {len(nodes)}")

Total chunks: 479


In [6]:
sample_node = nodes[100]
print("Metadata:", sample_node.metadata)
print("\n--- Chunk text ---\n")
print(sample_node.text)

Metadata: {'page_label': '33', 'file_name': 'records.pdf', 'file_path': 'c:\\Users\\koolr\\OneDrive\\Agentic AI June 2026\\Week 2 Project - RAG\\sports_rag_starter\\sports_rag\\sports_docs\\cricket\\records.pdf', 'file_type': 'application/pdf', 'file_size': 4754220, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14', 'sport': 'cricket', 'content_type': 'records'}

--- Chunk text ---

Most tournaments
Rank Tournaments Player Team Matches Period Ref
=1 6
Sachin Tendulkar
  India 45 1992–2011 [140]
Javed Miandad
  Pakistan 33 1975–1996 [141]
=3 5
Imran Khan 28 1975–1992 [142]
Arjuna Ranatunga
 Sri Lanka
30 1983–1999 [143]
Aravinda de Silva 35 1987–2003 [144]
Wasim Akram
  Pakistan 38 1987–2003 [145]
Inzamam-ul-Haq 35 1992–2007 [146]
Sanath Jayasuriya
  Sri Lanka 38 1992–2007 [147]
Brian Lara
  West Indies 34 1992–2007 [148]
Shivnarine Chanderpaul 31 1996–2011 [149]
Jacques Kallis
  South Africa 36 1996–2011 [150]
Muttiah Muralitharan
  Sri Lanka 40 1996–2011 [151]
Ricky Pon

In [7]:
import os
from dotenv import load_dotenv
from llama_index.embeddings.nebius import NebiusEmbedding

load_dotenv()

embed_model = NebiusEmbedding(
    model_name="BAAI/bge-en-icl",
    api_key=os.environ["NEBIUS_API_KEY"],
)

sample_text = "Brazil has won the FIFA World Cup five times."
vector = embed_model.get_text_embedding(sample_text)

print(f"Vector length (dimensions): {len(vector)}")
print(f"First 10 numbers: {vector[:10]}")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [8]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.environ.get("NEBIUS_API_KEY")
print("Key found:", key is not None)
print("Key length:", len(key) if key else 0)
print("First 4 characters:", key[:4] if key else None)

Key found: True
Key length: 235
First 4 characters: v1.C


In [9]:
import os
from dotenv import load_dotenv
from llama_index.embeddings.nebius import NebiusEmbedding

load_dotenv()

embed_model = NebiusEmbedding(
    model_name="BAAI/bge-en-icl",
    api_key=os.environ["NEBIUS_API_KEY"],
)

sample_text = "Brazil has won the FIFA World Cup five times."
vector = embed_model.get_text_embedding(sample_text)

print(f"Vector length (dimensions): {len(vector)}")
print(f"First 10 numbers: {vector[:10]}")

2026-06-14 16:51:02,838 - INFO - HTTP Request: POST https://api.studio.nebius.ai/v1/embeddings "HTTP/1.1 404 Not Found"


NotFoundError: Error code: 404 - {'detail': 'The model `BAAI/bge-en-icl` does not exist.'}

In [10]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

resp = requests.get(
    "https://api.tokenfactory.nebius.com/v1/models",
    headers={"Authorization": f"Bearer {os.environ['NEBIUS_API_KEY']}"}
)
resp.raise_for_status()
models = resp.json()["data"]

print(f"Total models available: {len(models)}\n")
print("Embedding models:")
for m in models:
    if "embed" in m["id"].lower():
        print(" -", m["id"])

Total models available: 34

Embedding models:
 - Qwen/Qwen3-Embedding-8B


In [11]:
import os
from dotenv import load_dotenv
from llama_index.embeddings.nebius import NebiusEmbedding

load_dotenv(override=True)

embed_model = NebiusEmbedding(
    model_name="Qwen/Qwen3-Embedding-8B",
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.tokenfactory.nebius.com/v1",
)

sample_text = "Brazil has won the FIFA World Cup five times."
vector = embed_model.get_text_embedding(sample_text)

print(f"Vector length (dimensions): {len(vector)}")
print(f"First 10 numbers: {vector[:10]}")

2026-06-14 16:54:13,113 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


Vector length (dimensions): 4096
First 10 numbers: [-0.004059693776071072, 0.047373268753290176, -0.011477028951048851, -0.01770392805337906, -0.012026460841298103, 0.027715804055333138, 0.014895718544721603, -0.008607772178947926, -0.01916908100247383, 0.018314408138394356]


In [12]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

chroma_client = chromadb.PersistentClient(path="./chroma_db")
test_collection = chroma_client.get_or_create_collection("test_collection")
vector_store = ChromaVectorStore(chroma_collection=test_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

test_index = VectorStoreIndex(nodes[:5], storage_context=storage_context, embed_model=embed_model)
print("Test collection count:", test_collection.count())

retriever = test_index.as_retriever(similarity_top_k=2)
results = retriever.retrieve("cricket world cup")
for r in results:
    print(r.metadata)

2026-06-14 16:58:15,330 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 16:58:15,868 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


Test collection count: 5
{'page_label': '1', 'file_name': 'history.pdf', 'file_path': 'c:\\Users\\koolr\\OneDrive\\Agentic AI June 2026\\Week 2 Project - RAG\\sports_rag_starter\\sports_rag\\sports_docs\\cricket\\history.pdf', 'file_type': 'application/pdf', 'file_size': 3076735, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14', 'sport': 'cricket', 'content_type': 'history'}
{'page_label': '1', 'file_name': 'history.pdf', 'file_path': 'c:\\Users\\koolr\\OneDrive\\Agentic AI June 2026\\Week 2 Project - RAG\\sports_rag_starter\\sports_rag\\sports_docs\\cricket\\history.pdf', 'file_type': 'application/pdf', 'file_size': 3076735, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14', 'sport': 'cricket', 'content_type': 'history'}


In [13]:
sports_collection = chroma_client.get_or_create_collection("sports_rag")
vector_store = ChromaVectorStore(chroma_collection=sports_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)
print("Total chunks indexed:", sports_collection.count())

2026-06-14 17:05:08,584 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:09,404 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:10,042 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:10,702 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:11,338 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:11,917 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:12,440 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:13,067 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:05:13,667 - INFO - HTTP Re

Total chunks indexed: 479


In [14]:
retriever = index.as_retriever(similarity_top_k=3)

def show_results(query):
    print(f"\n=== Query: {query} ===")
    results = retriever.retrieve(query)
    for r in results:
        print(f"\n[score: {r.score:.3f}] sport={r.metadata.get('sport')} "
              f"type={r.metadata.get('content_type')} "
              f"file={r.metadata.get('file_name')} page={r.metadata.get('page_label')}")
        print(r.text[:200].replace("\n", " ") + "...")

In [15]:
show_results("How many times has Brazil won the World Cup?")
show_results("What are the rules for leg before wicket in cricket?")
show_results("Which country dominates the most sports at World Cup level?")


=== Query: How many times has Brazil won the World Cup? ===


2026-06-14 17:10:31,149 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:10:31,424 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"



[score: 0.570] sport=soccer type=history file=history.pdf page=12
17 2002   South Korea  Japan  Brazil 2–0  Germany  Turkey 3–2  SouthKorea 32 18 2006   Germany  Italy 1–1(a.e.t.)(5–3 p)  France  Germany 3–1  Portugal 32 19 2010   South Africa  Spain 1–0(a.e.t.)  Ne...

[score: 0.539] sport=soccer type=records file=records.pdf page=21
"Five Stars: Brazil's FIFA World Cup Wins" (https://theanalyst.com/eu/2022/09/brazil-world-cup-wins-record/). The Analyst. Retrieved 2 January 2023. 35. Hayward, Ben (16 December 2022). "Which teams h...

[score: 0.507] sport=soccer type=records file=records.pdf page=21
27. Morrison, Neil (24 July 2014). "World Cup 2014 - Match Details" (https://www.rsssf.org/tables/2014full.html).RSSSF. Retrieved 14 December 2022. 28. "TECHNICAL REPORT - 2018 FIFA WORLD CUP RUSSIA" ...

=== Query: What are the rules for leg before wicket in cricket? ===

[score: 0.640] sport=cricket type=rules file=rules.pdf page=13
Altham, p. 95. 23. "Dates in cricket history" (https:/

2026-06-14 17:10:31,598 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"



[score: 0.493] sport=soccer type=history file=history.pdf page=12
17 2002   South Korea  Japan  Brazil 2–0  Germany  Turkey 3–2  SouthKorea 32 18 2006   Germany  Italy 1–1(a.e.t.)(5–3 p)  France  Germany 3–1  Portugal 32 19 2010   South Africa  Spain 1–0(a.e.t.)  Ne...

[score: 0.456] sport=soccer type=records file=records.pdf page=21
"Five Stars: Brazil's FIFA World Cup Wins" (https://theanalyst.com/eu/2022/09/brazil-world-cup-wins-record/). The Analyst. Retrieved 2 January 2023. 35. Hayward, Ben (16 December 2022). "Which teams h...

[score: 0.444] sport=soccer type=records file=records.pdf page=7
Rank Nation Gold Silver Bronze Total 1   Brazil 5 2 2 9 2   Germany 4 4 4 12 3   Italy 4 2 1 7 4   Argentina 3 3 0 6 5   France 2 2 2 6 6   Uruguay 2 0 0 2 7   England 1 0 0 1  Spain 1 0 0 1 9   Nethe...


In [ ]:
import json
from anthropic import Anthropic

claude_client = Anthropic()  # reads ANTHROPIC_API_KEY from env

ROUTER_SYSTEM_PROMPT = '''You are a query router for a sports knowledge RAG system covering Cricket, Soccer, and Rugby ONLY.

Given a user question, classify it and decide which sport(s) knowledge base(s) to search.

Categories:
- "rules": questions about how the game is played, laws, regulations, equipment
- "history": questions about World Cup results, historical events, records, statistics
- "comparison": questions comparing players, teams, or stats (within one sport)
- "cross_sport": questions that require comparing or combining information ACROSS cricket, soccer, and rugby
- "out_of_scope": the question is about a DIFFERENT sport (baseball, basketball, tennis, etc.) or unrelated to sports entirely (weather, general knowledge)

IMPORTANT: This system only covers cricket, soccer, and rugby. If a question asks something like "which sport" or "across sports" or "dominates the most sports" WITHOUT naming a specific sport outside this system's coverage, treat it as "cross_sport" -- interpret "sports" as referring to cricket, soccer, and rugby (the sports this system has), not sports in general.

For ALL categories EXCEPT "out_of_scope", produce a "sub_queries" object containing one focused, sport-specific reformulation of the question for EACH sport in "sports". Good reformulations use canonical, encyclopedic phrasing (the way a Wikipedia infobox or summary would describe it -- "most recent", "current holders", "most titles") instead of colloquial phrasing ("next", "last", "best").

Examples:
"When is the next cricket world cup?" -> {"category": "history", "sports": ["cricket"], "sub_queries": {"cricket": "What is the most recently announced or upcoming Cricket World Cup, and when is it scheduled?"}}
"When was the last Rugby World Cup?" -> {"category": "history", "sports": ["rugby"], "sub_queries": {"rugby": "Which country are the current holders of the Rugby World Cup, and when was it won?"}}
"What year was the latest soccer world cup?" -> {"category": "history", "sports": ["soccer"], "sub_queries": {"soccer": "When and where was the most recent FIFA World Cup held, and who won it?"}}
"Which country dominates the most sports at World Cup level?" -> {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"], "sub_queries": {"cricket": "Which country has won the most Cricket World Cups?", "soccer": "Which country has won the most Soccer World Cups?", "rugby": "Which country has won the most Rugby World Cup titles?"}}

For "out_of_scope", "sub_queries" should be {} and "sports" should be [].

Respond with ONLY a JSON object, nothing else -- no markdown code fences, no explanation, no backticks.
Format: {"category": "...", "sports": [...], "sub_queries": {...}}

For "rules", "history", and "comparison", "sports" should contain ONLY the relevant sport(s) -- usually just one.
For "cross_sport", "sports" should always be ["cricket", "soccer", "rugby"].
For "out_of_scope", "sports" should be [].
If a question doesn't name a sport but the category is rules/history/comparison, infer it from context (player names, terminology, etc.).'''

def route_query(question):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        system=ROUTER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    raw = response.content[0].text.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f"WARNING: could not parse router output: {raw}")
        return {"category": "history", "sports": ["cricket", "soccer", "rugby"]}

In [17]:
test_questions = [
    "How many times has Brazil won the World Cup?",
    "What are the rules for leg before wicket in cricket?",
    "Compare Messi vs Ronaldo's World Cup record",
    "Which country dominates the most sports at World Cup level?",
    "What is the offside rule?",
    "Who has scored the most rugby World Cup tries?",
]

for q in test_questions:
    print(q, "->", route_query(q))

2026-06-14 17:15:03,326 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


{"category": "history", "sports": ["soccer"]}
```
How many times has Brazil won the World Cup? -> {'category': 'history', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:15:03,921 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


{"category": "rules", "sports": ["cricket"]}
```
What are the rules for leg before wicket in cricket? -> {'category': 'history', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:15:04,489 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...

{"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"]}
```
Which country dominates the most sports at World Cup level? -> {'category': 'history', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:15:05,965 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


{"category": "rules", "sports": ["soccer"]}
```
What is the offside rule? -> {'category': 'history', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:15:06,544 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


{"category": "history", "sports": ["rugby"]}
```
Who has scored the most rugby World Cup tries? -> {'category': 'history', 'sports': ['cricket', 'soccer', 'rugby']}


In [18]:
import json
import re
from anthropic import Anthropic

claude_client = Anthropic()

ROUTER_SYSTEM_PROMPT = """You are a query router for a sports knowledge RAG system covering Cricket, Soccer, and Rugby ONLY.

Given a user question, classify it and decide which sport(s) knowledge base(s) to search.

Categories:
- "rules": questions about how the game is played, laws, regulations, equipment
- "history": questions about World Cup results, historical events, records, statistics
- "comparison": questions comparing players, teams, or stats (within one sport)
- "cross_sport": questions that require comparing or combining information ACROSS multiple sports (cricket, soccer, rugby)
- "out_of_scope": the question is not about cricket, soccer, or rugby at all (e.g. other sports, general knowledge, unrelated topics)

Respond with ONLY a JSON object, nothing else -- no markdown code fences, no explanation, no backticks.
Format: {"category": "rules|history|comparison|cross_sport|out_of_scope", "sports": ["cricket", "soccer", "rugby"]}

For "rules", "history", and "comparison", "sports" should contain ONLY the relevant sport(s) -- usually just one.
For "cross_sport", "sports" should always be ["cricket", "soccer", "rugby"].
For "out_of_scope", "sports" should be [].
If a question doesn't name a sport but the category is rules/history/comparison, infer it from context (player names, terminology, etc.)."""

def parse_router_response(raw):
    """Strip markdown code fences (```json ... ```) that Claude sometimes adds, then parse JSON."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE)
    return json.loads(cleaned.strip())

def route_query(question):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        system=ROUTER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    raw = response.content[0].text
    try:
        return parse_router_response(raw)
    except (json.JSONDecodeError, IndexError) as e:
        print(f"WARNING: could not parse router output: {raw!r} ({e})")
        return {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"]}

In [19]:
test_questions = [
    "How many times has Brazil won the World Cup?",
    "What are the rules for leg before wicket in cricket?",
    "Compare Messi vs Ronaldo's World Cup record",
    "Which country dominates the most sports at World Cup level?",
    "What is the offside rule?",
    "Who has scored the most rugby World Cup tries?",
    "Compare career stats of Babe Ruth and Mickey Mantle",
    "What's the weather like in Paris today?",
]

for q in test_questions:
    print(q, "->", route_query(q))

2026-06-14 17:18:03,274 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


How many times has Brazil won the World Cup? -> {'category': 'history', 'sports': ['soccer']}


2026-06-14 17:18:03,818 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


What are the rules for leg before wicket in cricket? -> {'category': 'rules', 'sports': ['cricket']}


2026-06-14 17:18:04,471 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...

Who has scored the most rugby World Cup tries? -> {'category': 'history', 'sports': ['rugby']}


2026-06-14 17:18:07,074 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Compare career stats of Babe Ruth and Mickey Mantle -> {'category': 'out_of_scope', 'sports': []}


2026-06-14 17:18:07,703 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


What's the weather like in Paris today? -> {'category': 'out_of_scope', 'sports': []}


In [20]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

def retrieve_for_query(question, top_k=3):
    """Routes the question, then retrieves chunks filtered by sport(s).
    Returns (route_info, list_of_NodeWithScore)."""
    route = route_query(question)
    category = route["category"]
    sports = route["sports"]

    if category == "out_of_scope":
        return route, []

    all_results = []
    for sport in sports:
        filters = MetadataFilters(filters=[ExactMatchFilter(key="sport", value=sport)])
        retriever = index.as_retriever(similarity_top_k=top_k, filters=filters)
        results = retriever.retrieve(question)
        all_results.extend(results)

    return route, all_results

In [21]:
for q in test_questions:
    route, results = retrieve_for_query(q)
    print(f"\n=== {q} ===")
    print(f"route: {route}")
    for r in results:
        print(f"  [{r.score:.3f}] sport={r.metadata.get('sport')} "
              f"type={r.metadata.get('content_type')} page={r.metadata.get('page_label')}")

2026-06-14 17:19:35,814 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:19:36,296 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"



=== How many times has Brazil won the World Cup? ===
route: {'category': 'history', 'sports': ['soccer']}
  [0.570] sport=soccer type=history page=12
  [0.539] sport=soccer type=records page=21
  [0.507] sport=soccer type=records page=21


2026-06-14 17:19:37,090 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:19:37,252 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"



=== What are the rules for leg before wicket in cricket? ===
route: {'category': 'rules', 'sports': ['cricket']}
  [0.640] sport=cricket type=rules page=13
  [0.595] sport=cricket type=rules page=11
  [0.587] sport=cricket type=rules page=15


2026-06-14 17:19:37,889 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:19:38,043 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


...


=== Who has scored the most rugby World Cup tries? ===
route: {'category': 'history', 'sports': ['rugby']}
  [0.578] sport=rugby type=history page=10
  [0.491] sport=rugby type=history page=10
  [0.453] sport=rugby type=history page=15


2026-06-14 17:19:42,016 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== Compare career stats of Babe Ruth and Mickey Mantle ===
route: {'category': 'out_of_scope', 'sports': []}


2026-06-14 17:19:42,988 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== What's the weather like in Paris today? ===
route: {'category': 'out_of_scope', 'sports': []}


In [22]:
SYNTHESIS_SYSTEM_PROMPT = """You are a sports knowledge assistant covering Cricket, Soccer, and Rugby.

You will be given a user question and a set of retrieved text chunks, each labeled with its source (sport, content type, page number).

Rules:
1. Answer ONLY using information present in the retrieved chunks. Do not use outside knowledge.
2. Some chunks may be bibliography/citation lists (URLs, "Retrieved from", author names, archive links) rather than substantive content -- ignore these when forming your answer, but you may still pull facts from chunks that mix substantive content with citation markers like [12].
3. If the chunks do not contain enough information to answer the question, say clearly: "I could not find this in the available sources." Do not guess or fill gaps with outside knowledge.
4. When you do answer, cite your sources inline using the format (sport, content_type, page N) after each claim.
5. For cross-sport questions, address each sport explicitly before giving an overall conclusion, and note if one or more sports' sources didn't have relevant information.
6. Be concise -- a few sentences to a short paragraph, not an essay."""

def format_chunks(results):
    if not results:
        return "No documents were retrieved."
    formatted = []
    for r in results:
        meta = r.metadata
        source = f"({meta.get('sport')}, {meta.get('content_type')}, page {meta.get('page_label')})"
        formatted.append(f"{source}:\n{r.text}")
    return "\n\n---\n\n".join(formatted)

def answer_query(question, top_k=3):
    route, results = retrieve_for_query(question, top_k=top_k)

    if route["category"] == "out_of_scope":
        return {
            "answer": "I can only answer questions about Cricket, Soccer, and Rugby (rules, history, and World Cup records). This question is outside that scope.",
            "route": route,
            "chunks_used": 0,
        }

    context = format_chunks(results)
    user_message = f"Question: {question}\n\nRetrieved context:\n\n{context}"

    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYNTHESIS_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )

    return {
        "answer": response.content[0].text,
        "route": route,
        "chunks_used": len(results),
    }

In [23]:
for q in test_questions + ["How many wickets did Shane Bond take in the 2007 World Cup?"]:
    result = answer_query(q)
    print(f"\n=== {q} ===")
    print(f"category: {result['route']['category']}, sports: {result['route']['sports']}, chunks: {result['chunks_used']}")
    print(result["answer"])

2026-06-14 17:22:06,790 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:22:07,317 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:22:08,930 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== How many times has Brazil won the World Cup? ===
category: history, sports: ['soccer'], chunks: 3
Brazil has won the World Cup **5 times** (soccer, history, page 12). 

The sources note that "With five titles, Brazil are the most successful World Cup team and also the only nation to have played in every World Cup (22) to date" (soccer, history, page 12). Brazil won their titles in 1958, 1962, 1970, 1994, and 2002 (soccer, history, page 12).


2026-06-14 17:22:09,654 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:22:09,904 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:22:11,492 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== What are the rules for leg before wicket in cricket? ===
category: rules, sports: ['cricket'], chunks: 3
According to the retrieved cricket rules:

**Leg Before Wicket (LBW)** occurs when:

1. The ball hits the batter without first hitting the bat, AND
2. The ball would have hit the wicket if the batter was not there, AND
3. The ball does not pitch on the leg side of the wicket

When these conditions are met, the batter is out.

**However, there is an exception:** If the ball strikes the batter outside the line of the off-stump AND the batter was attempting to play a stroke, the batter is not out. (cricket, rules, page 11)


2026-06-14 17:22:12,145 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:22:12,309 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:22:14,686 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...


=== Compare career stats of Babe Ruth and Mickey Mantle ===
category: out_of_scope, sports: [], chunks: 0
I can only answer questions about Cricket, Soccer, and Rugby (rules, history, and World Cup records). This question is outside that scope.


2026-06-14 17:22:24,912 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== What's the weather like in Paris today? ===
category: out_of_scope, sports: [], chunks: 0
I can only answer questions about Cricket, Soccer, and Rugby (rules, history, and World Cup records). This question is outside that scope.


2026-06-14 17:22:25,536 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:22:26,233 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:22:28,219 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== How many wickets did Shane Bond take in the 2007 World Cup? ===
category: history, sports: ['cricket'], chunks: 3
I could not find this in the available sources. While the retrieved chunks mention that Glenn McGrath secured 26 dismissals at the 2007 Cricket World Cup, there is no information about Shane Bond's wicket tally in that tournament.


In [24]:
ROUTER_SYSTEM_PROMPT = """You are a query router for a sports knowledge RAG system covering Cricket, Soccer, and Rugby ONLY.

Given a user question, classify it and decide which sport(s) knowledge base(s) to search.

Categories:
- "rules": questions about how the game is played, laws, regulations, equipment
- "history": questions about World Cup results, historical events, records, statistics
- "comparison": questions comparing players, teams, or stats (within one sport)
- "cross_sport": questions that require comparing or combining information ACROSS cricket, soccer, and rugby
- "out_of_scope": the question is about a DIFFERENT sport (baseball, basketball, tennis, etc.) or unrelated to sports entirely (weather, general knowledge)

IMPORTANT: This system only covers cricket, soccer, and rugby. If a question asks something like "which sport" or "across sports" or "dominates the most sports" WITHOUT naming a specific sport outside this system's coverage, treat it as "cross_sport" -- interpret "sports" as referring to cricket, soccer, and rugby (the sports this system has), not sports in general.
Example: "Which country dominates the most sports at World Cup level?" -> {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"]}
Example: "Compare stats across different sports for the best athletes" -> {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"]}

Respond with ONLY a JSON object, nothing else -- no markdown code fences, no explanation, no backticks.
Format: {"category": "rules|history|comparison|cross_sport|out_of_scope", "sports": ["cricket", "soccer", "rugby"]}

For "rules", "history", and "comparison", "sports" should contain ONLY the relevant sport(s) -- usually just one.
For "cross_sport", "sports" should always be ["cricket", "soccer", "rugby"].
For "out_of_scope", "sports" should be [].
If a question doesn't name a sport but the category is rules/history/comparison, infer it from context (player names, terminology, etc.)."""

def route_query(question):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        temperature=0,
        system=ROUTER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    raw = response.content[0].text
    try:
        return parse_router_response(raw)
    except (json.JSONDecodeError, IndexError) as e:
        print(f"WARNING: could not parse router output: {raw!r} ({e})")
        return {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"]}

In [25]:
for i in range(5):
    print(i, route_query("Which country dominates the most sports at World Cup level?"))

2026-06-14 17:25:20,678 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


0 {'category': 'cross_sport', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:25:21,359 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


1 {'category': 'cross_sport', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:25:21,981 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


2 {'category': 'cross_sport', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:25:22,735 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


3 {'category': 'cross_sport', 'sports': ['cricket', 'soccer', 'rugby']}


2026-06-14 17:25:23,702 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


4 {'category': 'cross_sport', 'sports': ['cricket', 'soccer', 'rugby']}


In [26]:
result = answer_query("Which country dominates the most sports at World Cup level?")
print(f"category: {result['route']['category']}, sports: {result['route']['sports']}, chunks: {result['chunks_used']}")
print(result["answer"])

2026-06-14 17:26:23,140 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:26:23,578 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:26:23,917 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:26:24,108 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:26:26,820 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


category: cross_sport, sports: ['cricket', 'soccer', 'rugby'], chunks: 9
I cannot fully answer this question based on the available sources. Here's what I found:

**Soccer:** Brazil dominates with 5 World Cup titles, more than any other nation (soccer, records, page 7). Germany has 4 titles, and Italy has 4 titles (soccer, history, page 12).

**Rugby:** New Zealand leads with 3 World Cup titles (rugby, history, page 3), followed by South Africa and Australia with 2 titles each (rugby, history, page 8).

**Cricket:** The retrieved sources contain only bibliographic citations and references, with no substantive information about which country has won the most Cricket World Cups.

**Overall conclusion:** I could not find comprehensive information about which country dominates across all three sports at World Cup level. To answer this properly, I would need data on cricket World Cup winners that isn't present in the available sources.


In [27]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

cricket_filter = MetadataFilters(filters=[ExactMatchFilter(key="sport", value="cricket")])
cricket_retriever = index.as_retriever(similarity_top_k=5, filters=cricket_filter)

# Try a more direct phrasing
for r in cricket_retriever.retrieve("Cricket World Cup winners list by country"):
    print(f"[{r.score:.3f}] type={r.metadata.get('content_type')} page={r.metadata.get('page_label')}")
    print(r.text[:250].replace("\n", " ") + "...\n")

2026-06-14 17:29:29,616 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


[0.784] type=history page=19
Retrieved from "https://en.wikipedia.org/w/index.php?title=Cricket_World_Cup&oldid=1358860928"...

[0.673] type=history page=8
Ed. Year Hosts Final Venue Final No. ofteamsChampions Result Runners-up 1 1975   England Lord's, London   WestIndies291/8 (60 overs) West Indies won by 17 runs(scorecard) (http://aus.cricinfo.com/db/ARCHIVE/WORLD_CUPS/WC75/WI_AUS_WC75_ODI-FINAL_21JUN...

[0.669] type=history page=8
Bridgetown  Australia281/4 (38 overs) Australia won by 53 runs(D/L)(scorecard) (http://content-aus.cricinfo.com/wc2007/engine/current/match/247507.html)  Sri Lanka215/8 (36 overs) 16 10 2011  India  Sri Lanka  Bangladesh WankhedeStadium,Mumbai  India...

[0.663] type=history page=12
Year Teams Total 1975   Australia,    East Africa,[b]    England,    India,    New Zealand,    Pakistan,    West Indies,    Sri Lanka 8 1979   Canada 1 1983   Zimbabwe 1 1987 none 0 1992   South Africa[a] 1 1996   Kenya,    Netherlands,    United Ara...

[0.646] type=records p

In [28]:
ROUTER_SYSTEM_PROMPT = """You are a query router for a sports knowledge RAG system covering Cricket, Soccer, and Rugby ONLY.

Given a user question, classify it and decide which sport(s) knowledge base(s) to search.

Categories:
- "rules": questions about how the game is played, laws, regulations, equipment
- "history": questions about World Cup results, historical events, records, statistics
- "comparison": questions comparing players, teams, or stats (within one sport)
- "cross_sport": questions that require comparing or combining information ACROSS cricket, soccer, and rugby
- "out_of_scope": the question is about a DIFFERENT sport (baseball, basketball, tennis, etc.) or unrelated to sports entirely (weather, general knowledge)

IMPORTANT: This system only covers cricket, soccer, and rugby. If a question asks something like "which sport" or "across sports" or "dominates the most sports" WITHOUT naming a specific sport outside this system's coverage, treat it as "cross_sport" -- interpret "sports" as referring to cricket, soccer, and rugby (the sports this system has), not sports in general.

For "cross_sport" questions ONLY, also produce a "sub_queries" object: for each of cricket, soccer, rugby, write a focused, sport-specific reformulation of the question that would retrieve the most relevant chunk for THAT sport (e.g. a "World Cup winners" question becomes "Which country has won the most Cricket World Cups?" / "...Soccer World Cups?" / "...Rugby World Cup titles?"). For all other categories, omit "sub_queries" or set it to {}.

Respond with ONLY a JSON object, nothing else -- no markdown code fences, no explanation, no backticks.
Format: {"category": "...", "sports": [...], "sub_queries": {"cricket": "...", "soccer": "...", "rugby": "..."}}

Example: "Which country dominates the most sports at World Cup level?" -> {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"], "sub_queries": {"cricket": "Which country has won the most Cricket World Cups?", "soccer": "Which country has won the most Soccer World Cups?", "rugby": "Which country has won the most Rugby World Cup titles?"}}

For "rules", "history", and "comparison", "sports" should contain ONLY the relevant sport(s) -- usually just one.
For "cross_sport", "sports" should always be ["cricket", "soccer", "rugby"].
For "out_of_scope", "sports" should be [].
If a question doesn't name a sport but the category is rules/history/comparison, infer it from context (player names, terminology, etc.)."""

def route_query(question):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        system=ROUTER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    raw = response.content[0].text
    try:
        return parse_router_response(raw)
    except (json.JSONDecodeError, IndexError) as e:
        print(f"WARNING: could not parse router output: {raw!r} ({e})")
        return {"category": "cross_sport", "sports": ["cricket", "soccer", "rugby"], "sub_queries": {}}

In [29]:
def retrieve_for_query(question, top_k=3):
    route = route_query(question)
    category = route["category"]
    sports = route["sports"]
    sub_queries = route.get("sub_queries", {}) or {}

    if category == "out_of_scope":
        return route, []

    all_results = []
    for sport in sports:
        query_text = sub_queries.get(sport, question) if category == "cross_sport" else question
        filters = MetadataFilters(filters=[ExactMatchFilter(key="sport", value=sport)])
        retriever = index.as_retriever(similarity_top_k=top_k, filters=filters)
        results = retriever.retrieve(query_text)
        all_results.extend(results)

    return route, all_results

In [30]:
result = answer_query("Which country dominates the most sports at World Cup level?")
print(f"category: {result['route']['category']}, sub_queries: {result['route'].get('sub_queries')}")
print(f"chunks: {result['chunks_used']}")
print(result["answer"])

2026-06-14 17:31:18,578 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:31:19,001 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:31:19,299 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:31:19,497 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:31:22,919 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


category: cross_sport, sub_queries: {'cricket': 'Which country has won the most Cricket World Cups?', 'soccer': 'Which country has won the most Soccer World Cups?', 'rugby': 'Which country has won the most Rugby World Cup titles?'}
chunks: 9
Based on the retrieved sources, I cannot fully answer this cross-sport question, as the available information is incomplete across sports.

**Cricket:** Australia dominates with 6 World Cup titles (1987, 1999, 2003, 2007, 2015, 2023), far ahead of India and West Indies with 2 titles each (cricket, history, page 10).

**Soccer:** Brazil dominates with 5 World Cup titles and is "the only nation to have played in every World Cup (22) to date" (soccer, history, page 12). Germany follows with 4 titles (soccer, records, page 7).

**Rugby:** South Africa leads with 4 World Cup titles (1995, 2007, 2019, 2023), followed by New Zealand with 3 titles (1987, 2011, 2015) (rugby, history, page 8).

**Overall:** I could not find comparative data across all three 

In [31]:
PLAIN_LLM_SYSTEM_PROMPT = """You are a helpful sports knowledge assistant covering Cricket, Soccer, and Rugby. Answer the user's question directly using your own knowledge. Be concise -- a few sentences to a short paragraph."""

def plain_llm_answer(question, temperature=0):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        temperature=temperature,
        system=PLAIN_LLM_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

In [32]:
eval_questions = [
    "How many times has Brazil won the World Cup?",
    "What are the rules for leg before wicket in cricket?",
    "Compare Messi vs Ronaldo's World Cup record",
    "Which country dominates the most sports at World Cup level?",
    "How many wickets did Shane Bond take in the 2007 World Cup?",
]

In [33]:
eval_results = []

for q in eval_questions:
    print(f"\n{'='*70}\nQUESTION: {q}\n{'='*70}")

    rag1 = answer_query(q)
    rag2 = answer_query(q)
    plain1 = plain_llm_answer(q)
    plain2 = plain_llm_answer(q)

    print("\n--- RAG (run 1) ---")
    print(rag1["answer"])
    print(f"(sources: {rag1['sources']})")

    print("\n--- RAG (run 2) ---")
    print(rag2["answer"])

    print("\n--- Plain LLM (run 1) ---")
    print(plain1)

    print("\n--- Plain LLM (run 2) ---")
    print(plain2)

    eval_results.append({
        "question": q, "rag1": rag1["answer"], "rag2": rag2["answer"],
        "plain1": plain1, "plain2": plain2, "sources": rag1["sources"],
    })

with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)


QUESTION: How many times has Brazil won the World Cup?


2026-06-14 17:33:15,675 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:33:16,121 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:33:17,516 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:33:18,389 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:33:18,567 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:33:19,832 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:33:20,678 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:33:21,579 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



--- RAG (run 1) ---
Brazil has won the World Cup **five times**, making them the most successful World Cup team in history (soccer, history, page 12). They were also the first nation to win the World Cup for the third time (1970), fourth time (1994), and fifth time (2002) (soccer, history, page 12).


KeyError: 'sources'

In [34]:
def answer_query(question, top_k=3):
    route, results = retrieve_for_query(question, top_k=top_k)

    sources = [
        {"sport": r.metadata.get("sport"),
         "content_type": r.metadata.get("content_type"),
         "page": r.metadata.get("page_label")}
        for r in results
    ]

    if route["category"] == "out_of_scope":
        return {
            "answer": "I can only answer questions about Cricket, Soccer, and Rugby (rules, history, and World Cup records). This question is outside that scope.",
            "route": route,
            "chunks_used": 0,
            "sources": [],
        }

    context = format_chunks(results)
    user_message = f"Question: {question}\n\nRetrieved context:\n\n{context}"

    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        temperature=0,
        system=SYNTHESIS_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )

    return {
        "answer": response.content[0].text,
        "route": route,
        "chunks_used": len(results),
        "sources": sources,
    }

In [35]:
PLAIN_LLM_SYSTEM_PROMPT = """You are a helpful sports knowledge assistant covering Cricket, Soccer, and Rugby. Answer the user's question directly using your own knowledge. Be concise -- a few sentences to a short paragraph."""

def plain_llm_answer(question, temperature=0):
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        temperature=temperature,
        system=PLAIN_LLM_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

In [36]:
eval_questions = [
    "How many times has Brazil won the World Cup?",
    "What are the rules for leg before wicket in cricket?",
    "Compare Messi vs Ronaldo's World Cup record",
    "Which country dominates the most sports at World Cup level?",
    "How many wickets did Shane Bond take in the 2007 World Cup?",
]

In [37]:
eval_results = []

for q in eval_questions:
    print(f"\n{'='*70}\nQUESTION: {q}\n{'='*70}")

    rag1 = answer_query(q)
    rag2 = answer_query(q)
    plain1 = plain_llm_answer(q)
    plain2 = plain_llm_answer(q)

    print("\n--- RAG (run 1) ---")
    print(rag1["answer"])
    print(f"(sources: {rag1['sources']})")

    print("\n--- RAG (run 2) ---")
    print(rag2["answer"])

    print("\n--- Plain LLM (run 1) ---")
    print(plain1)

    print("\n--- Plain LLM (run 2) ---")
    print(plain2)

    eval_results.append({
        "question": q, "rag1": rag1["answer"], "rag2": rag2["answer"],
        "plain1": plain1, "plain2": plain2, "sources": rag1["sources"],
    })

with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)


QUESTION: How many times has Brazil won the World Cup?


2026-06-14 17:35:21,475 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:21,863 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:23,123 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:23,909 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:24,079 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:27,938 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:28,788 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:29,603 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



--- RAG (run 1) ---
Brazil has won the World Cup **5 times**, making them the most successful World Cup team in history (soccer, history, page 12). They are also the only nation to have played in every World Cup tournament to date (soccer, history, page 12).
(sources: [{'sport': 'soccer', 'content_type': 'history', 'page': '12'}, {'sport': 'soccer', 'content_type': 'records', 'page': '21'}, {'sport': 'soccer', 'content_type': 'records', 'page': '21'}])

--- RAG (run 2) ---
Brazil has won the World Cup **5 times**, making them the most successful World Cup team in history (soccer, history, page 12). They are also the only nation to have played in every World Cup tournament to date (soccer, history, page 12).

--- Plain LLM (run 1) ---
Brazil has won the FIFA World Cup **5 times** (1958, 1962, 1970, 1994, and 2002), making them the most successful nation in World Cup history. They are the only team to have won the tournament five times.

--- Plain LLM (run 2) ---
Brazil has won the FIFA

2026-06-14 17:35:30,331 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:30,800 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:33,287 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:34,020 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:34,179 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:35,761 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:39,166 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:41,871 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



--- RAG (run 1) ---
# Leg Before Wicket (LBW) Rules in Cricket

According to the rules, a batter is out LBW if the following conditions are met (cricket, rules, page 11):

1. **The ball hits the batter without first hitting the bat**
2. **The ball would have hit the wicket** if the batter was not there
3. **The ball does not pitch on the leg side of the wicket**

However, there is an important exception: **if the ball strikes the batter outside the line of the off-stump AND the batter was attempting to play a stroke, the batter is not out** (cricket, rules, page 11).

In essence, LBW protects batters from being dismissed when they legitimately attempt to play a shot, but prevents them from using their body as an illegal shield to protect the wicket.
(sources: [{'sport': 'cricket', 'content_type': 'rules', 'page': '13'}, {'sport': 'cricket', 'content_type': 'rules', 'page': '11'}, {'sport': 'cricket', 'content_type': 'rules', 'page': '15'}])

--- RAG (run 2) ---
# Leg Before Wicket (LB

...


--- RAG (run 1) ---
I could not find this in the available sources.

While the retrieved chunks reference World Cup records for both Messi and Ronaldo (including mentions of "Messi and Ronaldo - World Cup Stats" and "The World Cup records Messi owns"), the actual substantive comparison data is not included in these excerpts. The chunks provided are primarily bibliography/citation lists and headers rather than the detailed statistics needed to compare their World Cup records.

To answer your question properly, I would need access to chunks containing their actual goal tallies, appearances, assists, and other World Cup performance metrics.
(sources: [{'sport': 'soccer', 'content_type': 'history', 'page': '20'}, {'sport': 'soccer', 'content_type': 'records', 'page': '24'}, {'sport': 'soccer', 'content_type': 'history', 'page': '29'}])

--- RAG (run 2) ---
I could not find this in the available sources.

While the retrieved chunks reference World Cup records for both Messi and Ronaldo (in

2026-06-14 17:35:56,583 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:35:57,023 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:57,314 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:35:57,487 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:36:00,285 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:01,152 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:01,323 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:36:01,627 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:36:01,816 - INFO - HTTP Request: POST https://api.tokenfactory


--- RAG (run 1) ---
I can answer this question for each sport based on the retrieved sources:

**Cricket:** Australia dominates, with 6 World Cup titles (1987, 1999, 2003, 2007, 2015, 2023) (cricket, history, page 10).

**Soccer:** Brazil dominates, with 5 World Cup titles and being "the only nation to have played in every World Cup (22) to date" (soccer, history, page 12).

**Rugby:** South Africa and New Zealand are the top performers. South Africa has won 4 titles (1995, 2007, 2019, 2023), while New Zealand has won 3 titles (1987, 2011, 2015) (rugby, history, page 8).

**Overall conclusion:** No single country dominates across all three sports at World Cup level. Australia leads in cricket, Brazil leads in soccer, and South Africa leads in rugby. If measuring by total World Cup victories across all three sports, the available sources do not provide enough information to make a definitive cross-sport comparison.
(sources: [{'sport': 'cricket', 'content_type': 'history', 'page': '10'

2026-06-14 17:36:11,275 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:11,756 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:36:13,181 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:14,661 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:15,037 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:36:16,638 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:17,627 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:36:18,965 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



--- RAG (run 1) ---
I could not find this in the available sources. The retrieved chunks contain information about Glenn McGrath taking 26 wickets at the 2007 World Cup (cricket, records, page 24), but they do not include any information about Shane Bond's wicket tally in that tournament.
(sources: [{'sport': 'cricket', 'content_type': 'records', 'page': '20'}, {'sport': 'cricket', 'content_type': 'records', 'page': '24'}, {'sport': 'cricket', 'content_type': 'records', 'page': '22'}])

--- RAG (run 2) ---
I could not find this in the available sources. The retrieved chunks contain information about Glenn McGrath taking 26 wickets at the 2007 World Cup (cricket, records, page 24), but they do not include any information about Shane Bond's wicket tally in that tournament.

--- Plain LLM (run 1) ---
Shane Bond took 13 wickets in the 2007 Cricket World Cup, finishing as the tournament's leading wicket-taker. He had an outstanding tournament for New Zealand, playing a key role in their ru

In [38]:
import os

CACHE_FILE = "query_cache.json"

def normalize_question(question):
    q = question.lower().strip()
    q = re.sub(r"[^\w\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

_cache = load_cache()

def answer_query_cached(question, top_k=3):
    key = normalize_question(question)
    if key in _cache:
        result = dict(_cache[key])
        result["from_cache"] = True
        return result

    result = answer_query(question, top_k=top_k)
    result["from_cache"] = False
    _cache[key] = {k: v for k, v in result.items() if k != "from_cache"}
    save_cache(_cache)
    return result

In [39]:
import time

q = "How many times has Brazil won the World Cup?"

start = time.time()
r1 = answer_query_cached(q)
print(f"Call 1: {time.time() - start:.3f}s, from_cache={r1['from_cache']}")

start = time.time()
r2 = answer_query_cached(q)
print(f"Call 2: {time.time() - start:.3f}s, from_cache={r2['from_cache']}")

print("\nAnswers match:", r1["answer"] == r2["answer"])

2026-06-14 17:51:24,008 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 17:51:24,596 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 17:51:26,007 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Call 1: 4.874s, from_cache=False
Call 2: 0.000s, from_cache=True

Answers match: True


In [40]:
rugby_filter = MetadataFilters(filters=[ExactMatchFilter(key="sport", value="rugby")])
rugby_retriever = index.as_retriever(similarity_top_k=5, filters=rugby_filter)

print("=== RAW question ===")
for r in rugby_retriever.retrieve("Can you tell me when was the last Rugby world cup"):
    print(f"[{r.score:.3f}] type={r.metadata.get('content_type')} page={r.metadata.get('page_label')}")
    print(r.text[:200].replace("\n", " ") + "...\n")

print("=== REWRITTEN question ===")
for r in rugby_retriever.retrieve("When was the most recent Rugby World Cup held?"):
    print(f"[{r.score:.3f}] type={r.metadata.get('content_type')} page={r.metadata.get('page_label')}")
    print(r.text[:200].replace("\n", " ") + "...\n")

=== RAW question ===


2026-06-14 19:29:41,810 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


[0.574] type=history page=15
Retrieved from "https://en.wikipedia.org/w/index.php?title=Rugby_World_Cup&oldid=1358766536"...

[0.532] type=history page=13
Retrieved 23 February 2007. 48. "Rugby World Cup Locations Confirmed Through to 2033" (https://www.rugbyworldcup.com/news/714742/rugby-world-cup-host-locations-confirmed-through-to-2033) (Press releas...

[0.515] type=history page=12
BBC Sport. Retrieved 6 June2021. 35. Aylwin, Michael (31 October 2015). "New Zealand 34-17 Australia: Rugby World Cup 2015 final player ratings| Rugby World Cup 2015" (https://www.theguardian.com/spor...

[0.510] type=history page=13
"The History of the Webb Ellis Cup" (http://www.skysport.co.nz/rugby-webb-ellis-cup/). Sky Sport NewZealand. Retrieved 13 February 2013. 44. "Official Website of the Rugby World Cup" (https://web.arch...

[0.506] type=history page=9
Team  1987  1991  1995  1999  2003  2007  2011  2015  2019  2023  2027  2031 Years  Argentina Q Q Q Q Q Q Q Q Q Q Q TBD 11  Australia Q Q Q Q Q 

2026-06-14 19:29:42,337 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


[0.560] type=history page=1
Rugby World Cup Upcoming tournament  2027 Men's Rugby World Cup Sport Rugby union Instituted 1987 Number of teams 24 Regions International (World Rugby) Holders   South Africa (2023) Most titles   Sou...

[0.553] type=history page=13
Retrieved 23 February 2007. 48. "Rugby World Cup Locations Confirmed Through to 2033" (https://www.rugbyworldcup.com/news/714742/rugby-world-cup-host-locations-confirmed-through-to-2033) (Press releas...

[0.544] type=history page=15
Retrieved from "https://en.wikipedia.org/w/index.php?title=Rugby_World_Cup&oldid=1358766536"...

[0.535] type=history page=4
A scrum between Samoa (in blue) and Wales (in red) during the 2011 World Cup The Webb Ellis Cup Japan's hosting of the 2019 World Cup marked the first time the tournament had been held outside the tra...

[0.533] type=history page=12
BBC Sport. Retrieved 6 June2021. 35. Aylwin, Michael (31 October 2015). "New Zealand 34-17 Australia: Rugby World Cup 2015 final player ratings| 

In [41]:
SYNTHESIS_SYSTEM_PROMPT = """You are a sports knowledge assistant covering Cricket, Soccer, and Rugby.

You will be given a user question and a set of retrieved text chunks, each labeled with its source (sport, content type, page number).

Rules:
1. Answer ONLY using information present in the retrieved chunks. Do not use outside knowledge.
2. Some chunks may be bibliography/citation lists (URLs, "Retrieved from", author names, archive links) rather than substantive content -- ignore these when forming your answer, but you may still pull facts from chunks that mix substantive content with citation markers like [12].
3. If the chunks do not contain enough information to answer the question, say clearly: "I could not find this in the available sources." Do not guess or fill gaps with outside knowledge.
4. When you do answer, cite your sources inline using the format (sport, content_type, page N) after each claim.
5. For cross-sport questions, address each sport explicitly before giving an overall conclusion, and note if one or more sports' sources didn't have relevant information.
6. Be concise -- a few sentences to a short paragraph, not an essay.
7. Some chunks contain compressed "infobox"-style data (short label-value pairs, e.g. "Holders South Africa (2023) Most titles New Zealand 3 Upcoming tournament 2027") rather than full sentences. Treat these labeled fields as facts and connect them to answer the question directly -- e.g. a "Holders" field with a year tells you who won the most recent tournament and when. This is using information present in the chunk, not outside knowledge."""

In [42]:
result = answer_query("Can you tell me when was the last Rugby world cup")
print(result["answer"])
for s in result["sources"]:
    print(s)

2026-06-14 19:31:24,416 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:31:24,959 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:31:27,378 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


I could not find this in the available sources. The retrieved chunks contain only bibliography and citation information (URLs, archive links, and reference citations) rather than substantive content about when the last Rugby World Cup took place. To answer your question about the most recent Rugby World Cup, I would need chunks with actual tournament information.
{'sport': 'rugby', 'content_type': 'history', 'page': '15'}
{'sport': 'rugby', 'content_type': 'history', 'page': '13'}
{'sport': 'rugby', 'content_type': 'history', 'page': '12'}


In [43]:
for q in ["Can you tell me when was the last Rugby world cup",
          "when is the next cricket world cup?"]:
    result = answer_query(q)
    print(f"\n=== {q} ===")
    print(result["answer"])
    for s in result["sources"]:
        print(" ", s)

2026-06-14 19:32:31,707 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:32:32,096 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:32:35,574 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== Can you tell me when was the last Rugby world cup ===
I could not find this in the available sources. The retrieved chunks contain only bibliography and citation information (URLs, archive links, and reference citations) rather than substantive content about when the last Rugby World Cup took place. While the chunks mention references to Rugby World Cup tournaments from 2015 and 2019, they don't provide the actual tournament dates or confirm which was the most recent.

To answer your question accurately, I would need chunks with substantive content about recent Rugby World Cup tournaments.
  {'sport': 'rugby', 'content_type': 'history', 'page': '15'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '13'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '12'}


2026-06-14 19:32:36,219 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:32:36,399 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:32:38,049 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== when is the next cricket world cup? ===
I could not find this in the available sources. The retrieved chunks contain bibliography/citation information and historical Cricket World Cup data, but they do not include information about when the next Cricket World Cup will be held. While one chunk mentions that "ICC announces World Cup schedule; 14 teams in 2027 and 2031," this appears to be only a citation header without the substantive details about the upcoming tournament date.
  {'sport': 'cricket', 'content_type': 'history', 'page': '19'}
  {'sport': 'cricket', 'content_type': 'history', 'page': '17'}
  {'sport': 'cricket', 'content_type': 'history', 'page': '8'}


In [44]:
8. Distinguish between citation NOISE (bare URLs, "Retrieved from...", "Archived from...", access dates, author names alone -- ignore these) and citation TITLES that state a fact (e.g. an article titled "ICC announces World Cup schedule: 14 teams in 2027 and 2031" -- the title itself is usable information, even though it appears in a numbered reference list). If a reference's title directly answers the question, use it and cite it normally.

SyntaxError: unterminated string literal (detected at line 1) (2756474027.py, line 1)

In [45]:
8. Distinguish between citation NOISE (bare URLs, "Retrieved from...", "Archived from...", access dates, author names alone -- ignore these) and citation TITLES that state a fact (e.g. an article titled "ICC announces World Cup schedule: 14 teams in 2027 and 2031" -- the title itself is usable information, even though it appears in a numbered reference list). If a reference's title directly answers the question, use it and cite it normally.

SyntaxError: unterminated string literal (detected at line 1) (2756474027.py, line 1)

In [46]:
SYNTHESIS_SYSTEM_PROMPT = """You are a sports knowledge assistant covering Cricket, Soccer, and Rugby.

You will be given a user question and a set of retrieved text chunks, each labeled with its source (sport, content type, page number).

Rules:
1. Answer ONLY using information present in the retrieved chunks. Do not use outside knowledge.
2. Some chunks may be bibliography/citation lists (URLs, "Retrieved from", author names, archive links) rather than substantive content -- ignore these when forming your answer, but you may still pull facts from chunks that mix substantive content with citation markers like [12].
3. If the chunks do not contain enough information to answer the question, say clearly: "I could not find this in the available sources." Do not guess or fill gaps with outside knowledge.
4. When you do answer, cite your sources inline using the format (sport, content_type, page N) after each claim.
5. For cross-sport questions, address each sport explicitly before giving an overall conclusion, and note if one or more sports' sources didn't have relevant information.
6. Be concise -- a few sentences to a short paragraph, not an essay.
7. Some chunks contain compressed "infobox"-style data (short label-value pairs, e.g. "Holders South Africa (2023) Most titles New Zealand 3 Upcoming tournament 2027") rather than full sentences. Treat these labeled fields as facts and connect them to answer the question directly -- e.g. a "Holders" field with a year tells you who won the most recent tournament and when. This is using information present in the chunk, not outside knowledge.
8. Distinguish between citation NOISE (bare URLs, "Retrieved from...", "Archived from...", access dates, author names alone -- ignore these) and citation TITLES that state a fact (e.g. an article titled "ICC announces World Cup schedule: 14 teams in 2027 and 2031" -- the title itself is usable information, even though it appears in a numbered reference list). If a reference's title directly answers the question, use it and cite it normally.

_IncompleteInputError: incomplete input (3223424576.py, line 1)

_IncompleteInputError: incomplete input (591958622.py, line 1)

In [48]:
SYNTHESIS_SYSTEM_PROMPT = '''You are a sports knowledge assistant covering Cricket, Soccer, and Rugby.

You will be given a user question and a set of retrieved text chunks, each labeled with its source (sport, content type, page number).

Rules:
1. Answer ONLY using information present in the retrieved chunks. Do not use outside knowledge.
2. Some chunks may be bibliography/citation lists (URLs, "Retrieved from", author names, archive links) rather than substantive content -- ignore these when forming your answer, but you may still pull facts from chunks that mix substantive content with citation markers like [12].
3. If the chunks do not contain enough information to answer the question, say clearly: "I could not find this in the available sources." Do not guess or fill gaps with outside knowledge.
4. When you do answer, cite your sources inline using the format (sport, content_type, page N) after each claim.
5. For cross-sport questions, address each sport explicitly before giving an overall conclusion, and note if one or more sports' sources didn't have relevant information.
6. Be concise -- a few sentences to a short paragraph, not an essay.
7. Some chunks contain compressed "infobox"-style data (short label-value pairs, e.g. "Holders South Africa (2023) Most titles New Zealand 3 Upcoming tournament 2027") rather than full sentences. Treat these labeled fields as facts and connect them to answer the question directly -- e.g. a "Holders" field with a year tells you who won the most recent tournament and when. This is using information present in the chunk, not outside knowledge.
8. Distinguish between citation NOISE (bare URLs, "Retrieved from...", "Archived from...", access dates, author names alone -- ignore these) and citation TITLES that state a fact (e.g. an article titled "ICC announces World Cup schedule: 14 teams in 2027 and 2031" -- the title itself is usable information, even though it appears in a numbered reference list). If a reference's title directly answers the question, use it and cite it normally.'''

In [49]:
for q in ["when is the next cricket world cup?",
          "Can you tell me when was the last Rugby world cup",
          "How many times has Brazil won the World Cup?",
          "What are the rules for leg before wicket in cricket?"]:
    result = answer_query(q)
    print(f"\n=== {q} ===")
    print(result["answer"])
    for s in result["sources"]:
        print(" ", s)

2026-06-14 19:37:37,368 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:37:37,799 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:37:39,165 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== when is the next cricket world cup? ===
Based on the retrieved sources, the next Cricket World Cup will be in **2027** (cricket, history, page 17). The ICC announced that the 2027 tournament will feature 14 teams, along with a subsequent 2031 World Cup also featuring 14 teams.
  {'sport': 'cricket', 'content_type': 'history', 'page': '19'}
  {'sport': 'cricket', 'content_type': 'history', 'page': '17'}
  {'sport': 'cricket', 'content_type': 'history', 'page': '8'}


2026-06-14 19:37:39,869 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:37:40,113 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:37:41,722 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== Can you tell me when was the last Rugby world cup ===
I could not find this in the available sources. The retrieved chunks contain only bibliography and citation information (URLs, retrieval dates, archive links) rather than substantive content about when the most recent Rugby World Cup took place. To answer your question about the last Rugby World Cup, I would need chunks with actual tournament details and dates.
  {'sport': 'rugby', 'content_type': 'history', 'page': '15'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '13'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '12'}


2026-06-14 19:37:42,395 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:37:42,561 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:37:43,968 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== How many times has Brazil won the World Cup? ===
Brazil has won the World Cup **5 times**, making them the most successful World Cup team in history. (soccer, history, page 12)

The sources indicate that Brazil won their titles in 1958, 1962, 1970, 1994, and 2002, and they are also the only nation to have played in every World Cup tournament to date. (soccer, history, page 12)
  {'sport': 'soccer', 'content_type': 'history', 'page': '12'}
  {'sport': 'soccer', 'content_type': 'records', 'page': '21'}
  {'sport': 'soccer', 'content_type': 'records', 'page': '21'}


2026-06-14 19:37:44,708 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:37:44,973 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:37:46,788 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== What are the rules for leg before wicket in cricket? ===
# Leg Before Wicket (LBW) Rules in Cricket

According to the Laws of Cricket, a batter is out leg before wicket if:

1. **The ball hits the batter's body** without first hitting the bat
2. **The ball would have hit the wicket** if the batter was not there
3. **The ball does not pitch on the leg side** of the wicket

However, there is an important exception: **if the ball strikes the batter outside the line of the off-stump while the batter was attempting to play a stroke, the batter is not out.** (cricket, rules, page 11)

This rule is formally codified as Law 36 in the MCC Laws of Cricket.
  {'sport': 'cricket', 'content_type': 'rules', 'page': '13'}
  {'sport': 'cricket', 'content_type': 'rules', 'page': '11'}
  {'sport': 'cricket', 'content_type': 'rules', 'page': '15'}


In [50]:
result = answer_query("Can you tell me when was the last Rugby world cup", top_k=5)
print(result["answer"])
for s in result["sources"]:
    print(" ", s)

2026-06-14 19:40:49,106 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 19:40:49,489 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 19:40:51,314 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


I could not find this in the available sources. The retrieved chunks contain bibliography entries, reference lists, and a qualification table showing which teams participated in various Rugby World Cups (including 2023), but they do not include substantive information stating when the most recent Rugby World Cup took place or who won it.

To answer your question accurately, I would need a source chunk that explicitly states the date and results of the most recent tournament.
  {'sport': 'rugby', 'content_type': 'history', 'page': '15'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '13'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '12'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '13'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '9'}


In [51]:
SYNTHESIS_SYSTEM_PROMPT = '''You are a sports knowledge assistant covering Cricket, Soccer, and Rugby.

You will be given a user question and a set of retrieved text chunks, each labeled with its source (sport, content type, page number).

Rules:
1. Answer ONLY using information present in the retrieved chunks. Do not use outside knowledge.
2. Some chunks may be bibliography/citation lists (URLs, "Retrieved from", author names, archive links) rather than substantive content -- ignore these when forming your answer, but you may still pull facts from chunks that mix substantive content with citation markers like [12].
3. If the chunks do not contain enough information to answer the question, say clearly: "I could not find this in the available sources." Do not guess or fill gaps with outside knowledge.
4. Do NOT include inline citations like "(sport, content_type, page N)" in your answer text -- sources are displayed separately to the user below the answer. Write in clean, natural prose without citation markers.
5. For cross-sport questions, address each sport explicitly before giving an overall conclusion, and note if one or more sports' sources didn't have relevant information.
6. Be concise -- a few sentences to a short paragraph, not an essay.
7. Some chunks contain compressed "infobox"-style data (short label-value pairs, e.g. "Holders South Africa (2023) Most titles New Zealand 3 Upcoming tournament 2027") rather than full sentences. Treat these labeled fields as facts and connect them to answer the question directly -- e.g. a "Holders" field with a year tells you who won the most recent tournament and when. This is using information present in the chunk, not outside knowledge.
8. Distinguish between citation NOISE (bare URLs, "Retrieved from...", "Archived from...", access dates, author names alone -- ignore these) and citation TITLES that state a fact (e.g. an article titled "ICC announces World Cup schedule: 14 teams in 2027 and 2031" -- the title itself is usable information, even though it appears in a numbered reference list). If a reference's title directly answers the question, use it and cite it normally.'''

In [52]:
for q in ["Can you tell me when was the last Rugby world cup",
          "How many times has Brazil won the World Cup?",
          "What are the rules for leg before wicket in cricket?"]:
    result = answer_query(q)
    print(f"\n=== {q} ===")
    print(result["answer"])
    for s in result["sources"]:
        print(" ", s)

2026-06-14 20:01:30,624 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 20:01:31,163 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 20:01:32,787 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== Can you tell me when was the last Rugby world cup ===
I could not find this in the available sources. The retrieved chunks contain only bibliography and citation information (URLs, archive links, and reference lists) rather than substantive content about when the most recent Rugby World Cup took place. To answer your question about the last Rugby World Cup, I would need chunks with actual tournament details and dates.
  {'sport': 'rugby', 'content_type': 'history', 'page': '15'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '13'}
  {'sport': 'rugby', 'content_type': 'history', 'page': '12'}


2026-06-14 20:01:34,969 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 20:01:35,212 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 20:01:36,501 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== How many times has Brazil won the World Cup? ===
Brazil has won the World Cup **five times**, making them the most successful World Cup team in history. They are also the only nation to have played in every World Cup tournament to date. Brazil were the first team to win the World Cup for the third time (1970), the fourth time (1994), and the fifth time (2002).
  {'sport': 'soccer', 'content_type': 'history', 'page': '12'}
  {'sport': 'soccer', 'content_type': 'records', 'page': '21'}
  {'sport': 'soccer', 'content_type': 'records', 'page': '21'}


2026-06-14 20:01:37,177 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-14 20:01:37,395 - INFO - HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-14 20:01:39,657 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



=== What are the rules for leg before wicket in cricket? ===
# Leg Before Wicket (LBW) Rules

According to cricket law, a batter is out leg before wicket if:

1. **The ball hits the batter's body** without first hitting the bat
2. **The ball would have hit the wicket** if the batter had not been in the way
3. **The ball did not pitch on the leg side** of the wicket

However, there is an important exception: **if the ball strikes the batter outside the line of the off-stump while the batter was attempting to play a stroke, the batter is not out.**

In essence, LBW protects batters from being dismissed when they legitimately attempt to play a shot but miss, while preventing them from using their body as an illegal shield to protect the wicket.
  {'sport': 'cricket', 'content_type': 'rules', 'page': '13'}
  {'sport': 'cricket', 'content_type': 'rules', 'page': '11'}
  {'sport': 'cricket', 'content_type': 'rules', 'page': '15'}
